# Testing Context Extraction with Claude

In [44]:
# imports
from anthropic import Anthropic
import os

## Test Examples

In [45]:
fileCode1 = """
# Globale Variablen
num_a = 10
num_b = 20
c = 30

# Funktionen
def func1():
    return num_a + helper1()

def helper1():
    return num_b + helper2()

def helper2():
    return 42

def dummy1():
    test = 5;
    return test
"""

functionToAnalyze1 = "func1"

fileCode2 = """
# Globale Variablen
limit = 10
offset = 3
factor = 2
unused = 999

# Funktionen
def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor

def dummy2():
    return unused
"""

functionToAnalyze2 = "func2"

fileCode3 = """
# Globale Variablen
threshold = 100
step = 3
start = 1

# Funktionen
def func3():
    value = start
    while value < threshold:
        value = helper3(value)
    return value

def helper3(x):
    return x + step

def dummy3():
    return threshold
"""

functionToAnalyze3 = "func3"

fileCode4 = """
# Globale Variablen
base = 0
decrement = 1

# Funktionen
def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)

def is_base(x):
    return x <= base

def helper_unused():
    return decrement
"""

functionToAnalyze4 = "func4"

fileCode5 = """
# Globale Variablen
MAX = 100
MIN = 1
STEP = 5
factor = 2
unused = 999

# Funktionen
def func5(n):
    total = 0
    for i in range(MIN, n, STEP):
        if i % 2 == 0:
            total += helper_even(i)
        else:
            total += helper_odd(i)
    while total < MAX:
        total = helper_increment(total)
    return total

def helper_even(x):
    return x * factor + helper_increment(x)

def helper_odd(x):
    if x % 3 == 0:
        return x // 2
    else:
        return x + factor

def helper_increment(value):
    if value >= MAX:
        return value
    return value + STEP

def dummy5():
    temp = 42
    return unused
"""

functionToAnalyze5 = "func5"

fileCode6 = """
# Globale Variablen
LIMIT = 50
OFFSET = 3
factor = 2
unused_global = 999

def ufunc7():
    return "I am not used"

# Funktionen
def func6(x):
    result = 0
    for i in range(x):
        if i % 2 == 0:
            result += even(i)
        elif i % 3 == 0:
            result += helper_multiple_three(i)
        else:
            result += helper_odd(i)
    return result

def even(n):
    return n * factor

def helper_odd(n):
    return n + OFFSET

def helper_multiple_three(n):
    if n > LIMIT:
        return LIMIT
    return n * 2

def dummy6():
    temp = 123
    return unused_global
"""

functionToAnalyze6 = "func6"

fileCode7 = """
# Globale Variablen
BASE = 10
STEP = 2
MAX = 100
factor = 3

# Funktionen
def dummy7():
    return 42  # Nicht genutzt

def func7(n):
    total = 0
    while n > 0:
        total += process_step(n)
        n -= STEP
    return finalize(total)

def process_step(value):
    if value % 5 == 0:
        return helper5(value)
    elif value % 3 == 0:
        return helper3(value)
    else:
        return value + factor

def helper5(v):
    return v // 5

def helper3(v):
    return v // 3

def finalize(result):
    return result % MAX

def unused_helper():
    return 0
"""

functionToAnalyze7 = "func7"

fileCode8 = """
# Globale Variablen
LIMIT = 1000
FACTOR = 3
OFFSET = 7
BASE = 5
unused_global = 999

# Funktionen
def dummy_unused1():
    return 0

def func8(n):
    result = BASE
    for i in range(n):
        if i % 2 == 0:
            result += helper_even(i)
        else:
            result += helper_odd(i)
        if result > LIMIT:
            result = cap_value(result)
    return finalize(result)

def helper_even(x):
    if x > OFFSET:
        return x * FACTOR
    else:
        return x + helper_nested(x)

def helper_odd(y):
    total = y
    while total < LIMIT:
        total += helper_even(total) // 2
        if total % 5 == 0:
            total -= helper_subtract(total)
    return total

def helper_nested(z):
    if z % 3 == 0:
        return z // 3 + helper_extra(z)
    return z + 1

def helper_extra(v):
    if v % 7 == 0:
        return v - 1
    return v + 2

def helper_subtract(val):
    return val - OFFSET

def cap_value(val):
    if val > LIMIT:
        return LIMIT
    return val

def finalize(final_result):
    return final_result % FACTOR

def dummy_unused2():
    temp = 123
    return unused_global
"""

functionToAnalyze8 = "func8"



In [46]:
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

models = client.models.list()
for m in models.data:
    print(m.id)

claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514
claude-3-5-haiku-20241022
claude-3-haiku-20240307


In [47]:
import anthropic

client = anthropic.Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

In [48]:
def extract_context_claude(full_code: str, target_fn: str):
    message = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=20000,
        temperature=0.0,
        system=(
            "You are a static program analysis assistant. "
            "You reason over Python source code precisely and deterministically."
        ),
        messages=[
            {
                "role": "user",
                "content": f"""
Full source code:
{full_code}

Target function:
{target_fn}

Task:
1. Identify ALL functions that are directly or indirectly called by the target function.
2. Identify ALL global variables used by those functions.
3. Extract ONLY:
   - the definitions of those global variables
   - the definitions of those functions
4. Sort the extracted functions topologically:
   - functions first
   - callers after callees

Rules:
- Output ONLY valid Python code
- Do NOT include ```python blocks
- Do NOT include unused functions or globals.
- Do NOT include comments, explanations, or markdown.
- Preserve exact Python syntax and indentation.
"""
            }
        ]
    )
    
    return message.content[0].text

In [49]:
fullFileCode = fileCode8
finalFunctionToAnalyse = functionToAnalyze8

context = extract_context_claude(fullFileCode, finalFunctionToAnalyse)

print(context)

LIMIT = 1000
FACTOR = 3
OFFSET = 7
BASE = 5

def helper_extra(v):
    if v % 7 == 0:
        return v - 1
    return v + 2

def helper_nested(z):
    if z % 3 == 0:
        return z // 3 + helper_extra(z)
    return z + 1

def helper_subtract(val):
    return val - OFFSET

def helper_even(x):
    if x > OFFSET:
        return x * FACTOR
    else:
        return x + helper_nested(x)

def helper_odd(y):
    total = y
    while total < LIMIT:
        total += helper_even(total) // 2
        if total % 5 == 0:
            total -= helper_subtract(total)
    return total

def cap_value(val):
    if val > LIMIT:
        return LIMIT
    return val

def finalize(final_result):
    return final_result % FACTOR

def func8(n):
    result = BASE
    for i in range(n):
        if i % 2 == 0:
            result += helper_even(i)
        else:
            result += helper_odd(i)
        if result > LIMIT:
            result = cap_value(result)
    return finalize(result)


In [50]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="max-height:400px; overflow:auto; border:1px solid #ccc; padding:10px; white-space:pre;">
{context}
</div>
"""))